# Phase 1 — Data Engineering & Synthetic Degradation

**Objective:** Build a clean *pristine* split (EyeQ-quality-filtered) and produce 9 synthetically degraded variants (3 types × 3 severity levels).

**Outputs (all under `Drive/Thesis/results/phase1_data_engineering/`):**
- `metrics/pristine_split.csv` — list of kept APTOS images
- `metrics/manifest_<type>_<level>.csv` — every degraded image with original label
- `samples/grid_<type>.png` — visual grid of clean vs L/M/H
- `plots/class_distribution.png` — pristine class balance

Local working copies of degraded images go to `/content/data/degraded/` (kept off Drive — too large to be worth syncing).

In [ ]:
# Re-mount Drive in case this is a fresh runtime
from google.colab import drive
drive.mount('/content/drive')

## Config (mirror of 00_setup.ipynb)

In [ ]:
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/Thesis')
RESULTS_ROOT = DRIVE_ROOT / 'results'
PHASE_DIR    = RESULTS_ROOT / 'phase1_data_engineering'
for sub in ('metrics', 'plots', 'samples', 'logs'):
    (PHASE_DIR / sub).mkdir(parents=True, exist_ok=True)

LOCAL_ROOT   = Path('/content/data')
APTOS_DIR    = LOCAL_ROOT / 'aptos'
EYEQ_DIR     = LOCAL_ROOT / 'eyeq'
DEGRADED_DIR = LOCAL_ROOT / 'degraded'
DEGRADED_DIR.mkdir(parents=True, exist_ok=True)

# --- EDIT THESE TWO LINES if your CSVs/image folders are named differently ---
APTOS_CSV       = next(APTOS_DIR.rglob('train.csv'))                 # APTOS labels
APTOS_IMAGES    = next(p for p in APTOS_DIR.rglob('train_images') if p.is_dir())
EYEQ_CSV        = next(EYEQ_DIR.rglob('*.csv'), None)                # EyeQ labels (optional)
# ----------------------------------------------------------------------------

# Class names + degradation grid
CLASS_NAMES        = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
DEGRADATION_TYPES  = ('blur', 'exposure', 'noise')
DEGRADATION_LEVELS = ('low', 'mid', 'high')
DEGRADATION_PARAMS = {
    'blur':     {'low': 2.0,  'mid': 5.0,  'high': 9.0},   # Gaussian sigma (px)
    'exposure': {'low': 0.7,  'mid': 0.4,  'high': 0.2},   # gain multiplier (<1 darker)
    'noise':    {'low': 0.02, 'mid': 0.06, 'high': 0.12},  # std as fraction of 255
}
EYEQ_QUALITY_LABELS = {0: 'good', 1: 'usable', 2: 'reject'}

print('APTOS csv   :', APTOS_CSV)
print('APTOS images:', APTOS_IMAGES)
print('EyeQ csv    :', EYEQ_CSV)

## 1. Quality filter (EyeQ → pristine subset)

If EyeQ has overlapping ids with APTOS we keep only `good` quality images. Otherwise we fall back to the full APTOS set (with a note in the log).

In [ ]:
import pandas as pd

def filter_pristine(aptos_csv, eyeq_csv, out_csv,
                    keep_quality=('good',),
                    eyeq_id_col='image', eyeq_quality_col='quality',
                    aptos_id_col='id_code'):
    aptos = pd.read_csv(aptos_csv)
    if eyeq_csv is None or not Path(eyeq_csv).exists():
        print('[fallback] No EyeQ CSV found — keeping all APTOS rows as pristine.')
        aptos.to_csv(out_csv, index=False)
        return aptos, 'fallback'
    eyeq = pd.read_csv(eyeq_csv)
    if pd.api.types.is_integer_dtype(eyeq[eyeq_quality_col]):
        eyeq['__q'] = eyeq[eyeq_quality_col].map(EYEQ_QUALITY_LABELS)
    else:
        eyeq['__q'] = eyeq[eyeq_quality_col].astype(str).str.lower()
    keep_ids = set(eyeq.loc[eyeq['__q'].isin(keep_quality), eyeq_id_col].astype(str))
    pristine = aptos[aptos[aptos_id_col].astype(str).isin(keep_ids)].reset_index(drop=True)
    if len(pristine) == 0:
        print('[warn] EyeQ filter removed all rows — falling back to full APTOS.')
        aptos.to_csv(out_csv, index=False)
        return aptos, 'fallback'
    pristine.to_csv(out_csv, index=False)
    return pristine, 'eyeq'

pristine_csv = PHASE_DIR / 'metrics' / 'pristine_split.csv'
pristine_df, mode = filter_pristine(APTOS_CSV, EYEQ_CSV, pristine_csv)
print(f'Mode: {mode} | kept {len(pristine_df)} of {pd.read_csv(APTOS_CSV).shape[0]} images')
pristine_df.head()

## 2. Class balance plot (saved)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

counts = pristine_df['diagnosis'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([CLASS_NAMES[i] for i in counts.index], counts.values, color='steelblue')
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width()/2, v, str(v), ha='center', va='bottom', fontsize=9)
ax.set_title('Pristine class distribution (Phase 1 input to Phase 2)')
ax.set_ylabel('Image count')
plt.xticks(rotation=15)
plt.tight_layout()
out = PHASE_DIR / 'plots' / 'class_distribution.png'
plt.savefig(out, dpi=150)
plt.show()
print('Saved:', out)

## 3. Synthetic degradation functions

In [ ]:
import cv2
import numpy as np
from PIL import Image

def _np(img):  return np.array(img.convert('RGB'))
def _pil(arr): return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def gaussian_blur(img, sigma):
    k = max(3, int(2 * round(3 * sigma) + 1))
    return _pil(cv2.GaussianBlur(_np(img), (k, k), sigmaX=sigma, sigmaY=sigma))

def exposure_shift(img, gain):
    return _pil(_np(img).astype(np.float32) * gain)

def gaussian_noise(img, std_frac):
    arr = _np(img).astype(np.float32)
    return _pil(arr + np.random.normal(0, std_frac * 255, arr.shape))

DEGRADERS = {'blur': gaussian_blur, 'exposure': exposure_shift, 'noise': gaussian_noise}

def apply_degradation(img, kind, level):
    return DEGRADERS[kind](img, DEGRADATION_PARAMS[kind][level])

## 4. Generate every (type, level) variant

Writes images to `/content/data/degraded/<type>/<level>/` and a manifest CSV per split. The manifest is also copied to Drive for archiving.

In [ ]:
from tqdm.auto import tqdm
import shutil

def build_split(kind, level):
    out_dir = DEGRADED_DIR / kind / level
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for _, row in tqdm(pristine_df.iterrows(), total=len(pristine_df),
                       desc=f'{kind}-{level}', leave=False):
        src = APTOS_IMAGES / f"{row['id_code']}.png"
        if not src.exists():
            continue
        out_img = apply_degradation(Image.open(src).convert('RGB'), kind, level)
        out_img.save(out_dir / f"{row['id_code']}.png")
        rows.append({'id_code': row['id_code'], 'diagnosis': int(row['diagnosis']),
                     'degradation': kind, 'level': level,
                     'rel_path': f"{row['id_code']}.png"})
    manifest = pd.DataFrame(rows)
    local_csv = out_dir / 'manifest.csv'
    manifest.to_csv(local_csv, index=False)
    drive_csv = PHASE_DIR / 'metrics' / f'manifest_{kind}_{level}.csv'
    shutil.copy(local_csv, drive_csv)
    return manifest

for k in DEGRADATION_TYPES:
    for l in DEGRADATION_LEVELS:
        m = build_split(k, l)
        print(f'  {k}/{l}: {len(m)} images')

## 5. Visual sanity check — sample grid per degradation type

Four rows = four random images. Columns = clean / low / mid / high. Saved one PNG per degradation type for the appendix.

In [ ]:
rng = np.random.default_rng(42)
sample_ids = rng.choice(pristine_df['id_code'].values, size=4, replace=False)

for kind in DEGRADATION_TYPES:
    fig, axes = plt.subplots(4, 4, figsize=(11, 11))
    for r, sid in enumerate(sample_ids):
        clean = Image.open(APTOS_IMAGES / f'{sid}.png').convert('RGB').resize((224, 224))
        axes[r, 0].imshow(clean); axes[r, 0].set_title('clean' if r == 0 else '')
        for c, lvl in enumerate(DEGRADATION_LEVELS, start=1):
            deg = apply_degradation(clean, kind, lvl)
            axes[r, c].imshow(deg)
            if r == 0:
                axes[r, c].set_title(lvl)
        for ax in axes[r]:
            ax.axis('off')
    fig.suptitle(f'Degradation: {kind}', fontsize=14)
    plt.tight_layout()
    out = PHASE_DIR / 'samples' / f'grid_{kind}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved:', out)

## 6. Phase 1 manifest summary

In [ ]:
rows = []
for k in DEGRADATION_TYPES:
    for l in DEGRADATION_LEVELS:
        n = len(list((DEGRADED_DIR / k / l).glob('*.png')))
        rows.append({'degradation': k, 'level': l, 'n_images': n})
summary = pd.DataFrame(rows)
summary_path = PHASE_DIR / 'metrics' / 'phase1_summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved:', summary_path)
summary

---
**Phase 1 complete.** Proceed to `02_phase2_model_benchmarking.ipynb`.